# Chapter 4 — VLM Forward Pass

This notebook runs a **full forward pass** through nanochat_vlm.

Pipeline:
    image → vision encoder → projector
    +
    text → tokenizer → embeddings
    →
    fusion → transformer → logits

No training. Inference only.

## What This Notebook Proves

We verify that:
- dimensions line up
- attention masking is correct
- multimodal inputs produce logits

We do NOT:
- compute loss
- backprop
- update weights

In [1]:
import torch
import torch.nn as nn

## Model Components

Assumed components:

1. Vision encoder
2. Vision → LM projector
3. Language model (nanochat transformer)

Each is treated as a black box here.

In [2]:
BATCH = 2
NUM_PATCHES = 256
TEXT_LEN = 16

VISION_DIM = 1024
LM_DIM = 2048
VOCAB_SIZE = 65_536

## Vision Encoder (Mock)

Outputs patch embeddings:
    (B, N_patches, D_v)

In [3]:
class DummyVisionEncoder(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.out_dim = out_dim

    def forward(self, images):
        B = images.shape[0]
        return torch.randn(B, NUM_PATCHES, self.out_dim)

## Vision → LM Projector

In [4]:
class LinearProjector(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.proj(x)

## Language Model (Mock)

Consumes:
- embeddings
- attention mask

Outputs:
- logits over vocabulary

In [5]:
class DummyLM(nn.Module):
    def __init__(self, embed_dim, vocab_size):
        super().__init__()
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, embeds, attn_mask=None):
        return self.lm_head(embeds)

## Inputs

- image tensor (placeholder)
- text embeddings (already embedded)

In [6]:
images = torch.randn(BATCH, 3, 224, 224)
text_embeds = torch.randn(BATCH, TEXT_LEN, LM_DIM)

## Vision Path

In [7]:
vision_encoder = DummyVisionEncoder(VISION_DIM)
projector = LinearProjector(VISION_DIM, LM_DIM)

vision_feats = vision_encoder(images)
vision_embeds = projector(vision_feats)

print("Vision embeds:", vision_embeds.shape)

Vision embeds: torch.Size([2, 256, 2048])


## Image Boundary Tokens

In [8]:
im_start = torch.randn(BATCH, 1, LM_DIM)
im_end   = torch.randn(BATCH, 1, LM_DIM)

image_block = torch.cat(
    [im_start, vision_embeds, im_end],
    dim=1
)

## Multimodal Fusion

In [9]:
joint_embeds = torch.cat(
    [image_block, text_embeds],
    dim=1
)

L = joint_embeds.shape[1]
print("Joint sequence:", joint_embeds.shape)

Joint sequence: torch.Size([2, 274, 2048])


## Attention Mask

Standard causal mask works because images are prepended.

In [10]:
attn_mask = torch.tril(torch.ones(L, L)).bool()
attn_mask = attn_mask.unsqueeze(0).expand(BATCH, -1, -1)

## Forward Pass Through LM

In [11]:
lm = DummyLM(LM_DIM, VOCAB_SIZE)

logits = lm(joint_embeds, attn_mask=attn_mask)

print("Logits shape:", logits.shape)

Logits shape: torch.Size([2, 274, 65536])


## Text Token Logits

Only text positions are supervised later.

In [12]:
image_len = image_block.shape[1]
text_logits = logits[:, image_len:, :]

print("Text logits:", text_logits.shape)

Text logits: torch.Size([2, 16, 65536])


## Guarantees

This notebook confirms:
- end-to-end multimodal forward works
- tensor shapes are correct
- LM sees images as prefix context